# 🎓 SmartEvent Strathmore University — Machine Learning Pipeline (Google Colab)
**Author:** Elvis Wanyama Wafuke (154210)  
**Supervisor:** Mr. Allan Vikiru  
**Institution:** Strathmore University — School of Computing and Engineering Sciences (SCES)  

---
### Pipeline Stages Included in This Notebook:
1. **Environment Setup & Repository Sync** (Pulls the `smartevent` ML package or runs standalone)
2. **Stage 1 — Kaggle Dataset Audit & Leakage Removal** (`kaggle_prep.py`)
3. **Stage 2 — Synthetic Strathmore University Event Registration Generation** (`synthetic.py`)
4. **Stage 3 — Deterministic Data Cleaning & Imputation Prep** (`clean.py`)
5. **Stage 4 & 5 — Chronological Event Split (560 Train / 140 Future Test Events) & Feature Engineering**
6. **Stage 6 & 7 — 5-Fold GroupKFold Model Comparison & 5-Stage Logistic Regression Refinement**
7. **Stage 8 — Inline Evaluation Visualizations (ROC Curves, Calibration, Feature Coefficients, Headcount Scatter)**
8. **Live Interactive Attendance & Africa's Talking Priority SMS Boost Prediction**

In [ ]:
# =============================================================================
# CELL 1: Environment Setup & SmartEvent ML Package Initialization (Google Colab)
# =============================================================================
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd

# If running inside local repo, switch to ml-service directory
if Path('ml-service/smartevent_ml').exists():
    os.chdir('ml-service')
if '.' not in sys.path:
    sys.path.insert(0, '.')

# Ensure smartevent_ml package exists in Google Colab even if GitHub repo is private
pkg_dir = Path('smartevent_ml')
pkg_dir.mkdir(exist_ok=True)
_PKG_FILES = {'__init__.py': '"""SmartEvent attendance-prediction package (data prep, training and inference)."""\n', 'clean.py': '"""\nStage 3 - Clean the raw registration export.\n\nOnly *deterministic* row-level fixes happen here (they do not learn anything from the data,\nso they cannot leak test information):\n  - remove double submissions, standardise spellings, convert impossible values to missing,\n  - drop rows whose outcome was never recorded.\nImputation and normalisation are statistics learned from data, so they live inside the\nsklearn Pipeline and are fitted on the training split only (see train.py).\n"""\nimport json\nimport re\n\nimport numpy as np\nimport pandas as pd\n\nfrom .config import CATEGORIES, PROCESSED_DIR, TARGET\n\nCATEGORY_ALIASES = {\n    "academic": "Academic", "workshop": "Workshop", "work shop": "Workshop", "seminar": "Seminar",\n    "sports": "Sports", "sport": "Sports", "career fair": "Career Fair", "careers fair": "Career Fair",\n    "social & cultural": "Social & Cultural", "social and cultural": "Social & Cultural", "cultural": "Social & Cultural",\n}\nSLOT_ALIASES = {"morning": "Morning", "am": "Morning", "afternoon": "Afternoon", "pm": "Afternoon",\n                "evening": "Evening", "eve": "Evening"}\nSMS_ALIASES = {"none": "none", "0": "none", "": "none", "nan": "none", "low": "low", "1": "low",\n               "medium": "medium", "med": "medium", "2": "medium", "high": "high", "3": "high"}\nRANGES = {"travel_time_minutes": (0, 240), "venue_capacity": (1, 5000), "lead_days": (0, 365),\n          "year_of_study": (1, 6), "day_of_week": (0, 6)}\n\n\ndef _norm(s: pd.Series) -> pd.Series:\n    return s.astype(str).str.strip().str.lower().map(lambda x: re.sub(r"\\s+", " ", x))\n\n\ndef clean(df: pd.DataFrame) -> tuple[pd.DataFrame, list[dict]]:\n    log = []\n\n    def record(step, before, after, detail):\n        log.append({"step": step, "rows_before": int(before), "rows_after": int(after), "detail": detail})\n\n    n = len(df)\n    df = df.drop_duplicates(subset=["registration_id"], keep="first").copy()\n    record("remove double submissions", n, len(df), "duplicate registration_id (same student, same event)")\n\n    fixes = {}\n    before = df["event_category"].copy()\n    df["event_category"] = _norm(df["event_category"]).map(CATEGORY_ALIASES)\n    fixes["event_category"] = int((before != df["event_category"]).sum())\n    before = df["time_slot"].copy()\n    df["time_slot"] = _norm(df["time_slot"]).map(SLOT_ALIASES)\n    fixes["time_slot"] = int((before != df["time_slot"]).sum())\n    before = df["sms_priority"].copy()\n    df["sms_priority"] = _norm(df["sms_priority"].fillna("")).map(SMS_ALIASES)\n    fixes["sms_priority"] = int((before != df["sms_priority"]).sum())\n    df["weather"] = _norm(df["weather"]).replace({"nan": np.nan})\n    record("standardise spellings", len(df), len(df), f"values rewritten to canonical form: {fixes}")\n\n    n = len(df)\n    unknown = df["event_category"].isna() | df["time_slot"].isna() | ~df["event_category"].isin(CATEGORIES)\n    df = df[~unknown]\n    record("drop unmappable categories", n, len(df), "category/time slot not recognisable after standardising")\n\n    invalid = {}\n    for col, (lo, hi) in RANGES.items():\n        bad = df[col].notna() & ~df[col].between(lo, hi)\n        invalid[col] = int(bad.sum())\n        df.loc[bad, col] = np.nan\n    record("impossible values -> missing", len(df), len(df), f"out-of-range values set to NaN for imputation: {invalid}")\n\n    # venue_capacity is an event attribute: recover it from other rows of the same event when possible.\n    missing_cap = int(df["venue_capacity"].isna().sum())\n    df["venue_capacity"] = df.groupby("event_id")["venue_capacity"].transform(lambda s: s.fillna(s.median()))\n    record("recover venue capacity", len(df), len(df),\n           f"{missing_cap} missing capacities filled from the same event\'s other registrations")\n\n    n = len(df)\n    df = df[df[TARGET].notna()]\n    df[TARGET] = df[TARGET].astype(int)\n    record("drop unrecorded outcomes", n, len(df), "a row without the label cannot be used for supervised training")\n\n    df["event_date"] = pd.to_datetime(df["event_date"])\n    df = df.sort_values(["event_date", "event_id", "registration_id"]).reset_index(drop=True)\n    return df, log\n\n\ndef missing_summary(df: pd.DataFrame) -> dict:\n    m = df.isna().sum()\n    return {k: int(v) for k, v in m[m > 0].items()}\n\n\ndef run(verbose: bool = True) -> pd.DataFrame:\n    raw_path = PROCESSED_DIR.parent / "synthetic" / "raw_registrations.csv"\n    raw = pd.read_csv(raw_path, keep_default_na=True)\n    cleaned, log = clean(raw)\n    remaining = missing_summary(cleaned)\n    cleaned.to_csv(PROCESSED_DIR / "clean_registrations.csv", index=False)\n    (PROCESSED_DIR / "cleaning_report.json").write_text(json.dumps({\n        "raw_rows": int(len(raw)), "clean_rows": int(len(cleaned)), "log": log,\n        "missing_left_for_pipeline_imputation": remaining,\n        "note": "past_attendance_rate is missing by design for a student\'s first event (no history yet).",\n    }, indent=2))\n    if verbose:\n        print(f"[stage 3] Cleaning: {len(raw):,} raw rows -> {len(cleaned):,} clean rows")\n        for s in log:\n            print(f"          - {s[\'step\']}: {s[\'rows_before\']:,} -> {s[\'rows_after\']:,}  ({s[\'detail\']})")\n        print(f"[stage 3] Missing values left for in-pipeline imputation: {remaining}")\n    return cleaned\n\n\nif __name__ == "__main__":\n    run()\n', 'config.py': '"""Central configuration for the SmartEvent attendance-prediction pipeline."""\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parent.parent\nDATA_DIR = ROOT / "data"\nRAW_KAGGLE_CSV = DATA_DIR / "Attendance_Prediction.csv"\nPROCESSED_DIR = DATA_DIR / "processed"\nMODELS_DIR = ROOT / "models"\nREPORTS_DIR = ROOT / "reports"\nFIGURES_DIR = REPORTS_DIR / "figures"\n\nMODEL_PATH = MODELS_DIR / "attendance_model.joblib"\nMETADATA_PATH = MODELS_DIR / "model_metadata.json"\n\nSEED = 42\n\n# Vocabularies shared by the generator, the cleaner, the API and the web/mobile apps.\nCATEGORIES = ["Academic", "Workshop", "Seminar", "Sports", "Career Fair", "Social & Cultural"]\nTIME_SLOTS = ["Morning", "Afternoon", "Evening"]\nWEATHER = ["sunny", "cloudy", "rainy", "hot", "cold"]\nSMS_PRIORITIES = ["none", "low", "medium", "high"]\nDAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]\n\nTARGET = "attended"\nID_COLUMNS = ["registration_id", "event_id", "student_id", "event_date"]\n\n# Raw candidate predictors available at prediction time.\nCATEGORICAL_FEATURES = ["event_category", "time_slot", "weather", "sms_priority"]\nNUMERIC_FEATURES = [\n    "day_of_week", "venue_capacity", "registered_count", "lead_days", "exam_period",\n    "year_of_study", "hostel_resident", "travel_time_minutes",\n    "past_attendance_rate", "past_events_count",\n]\n# Columns carried over from the Kaggle student profiles that have no real effect.\n# They are deliberately kept as candidates so feature selection has to prove they are useless.\nNOISE_CANDIDATES = ["gender", "parent_education"]\n\n# Features created inside the sklearn pipeline (see features.add_engineered_features).\nENGINEERED_FEATURES = ["fill_ratio", "overbooked_excess", "is_weekend", "log_lead_days", "has_history"]\n', 'features.py': '"""\nFeature engineering + preprocessing pipeline (imputation, normalisation, encoding).\n\nEverything here is wrapped in sklearn objects so that the exact same transformations run\nduring training, during evaluation and inside the FastAPI service.\n"""\nimport numpy as np\nimport pandas as pd\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler\n\nRAW_CATEGORICAL = ["event_category", "time_slot", "weather", "sms_priority"]\nRAW_NUMERIC = ["day_of_week", "venue_capacity", "registered_count", "lead_days", "exam_period",\n               "year_of_study", "hostel_resident", "travel_time_minutes",\n               "past_attendance_rate", "past_events_count"]\nNOISE = ["gender", "parent_education"]          # no true effect; feature selection should drop them\n\nENGINEERED_NUMERIC = ["fill_ratio", "overbooked_excess", "is_weekend", "log_lead_days",\n                      "has_history", "evening_travel", "evening_hostel"]\n\n# Columns the API must supply (the engineered ones are derived automatically).\nINPUT_COLUMNS = RAW_CATEGORICAL + RAW_NUMERIC + NOISE\n\n\ndef add_engineered_features(X: pd.DataFrame) -> pd.DataFrame:\n    X = X.copy()\n    cap = X["venue_capacity"].replace(0, np.nan)\n    X["fill_ratio"] = X["registered_count"] / cap\n    X["overbooked_excess"] = (X["fill_ratio"] - 1).clip(lower=0)\n    X["is_weekend"] = (X["day_of_week"] >= 5).astype(float)\n    X["log_lead_days"] = np.log1p(X["lead_days"].clip(lower=0))\n    X["has_history"] = (X["past_events_count"].fillna(0) > 0).astype(float)\n    evening = (X["time_slot"] == "Evening").astype(float)\n    X["evening_travel"] = evening * X["travel_time_minutes"]\n    X["evening_hostel"] = evening * X["hostel_resident"]\n    return X\n\n\ndef engineered_names(_transformer, input_features):\n    return list(input_features) + ENGINEERED_NUMERIC\n\n\ndef make_preprocessor(numeric, categorical, scale: bool = True) -> ColumnTransformer:\n    num_steps = [("impute", SimpleImputer(strategy="median"))]\n    if scale:\n        num_steps.append(("scale", StandardScaler()))\n    cat_steps = [("impute", SimpleImputer(strategy="most_frequent")),\n                 ("onehot", OneHotEncoder(handle_unknown="ignore"))]\n    return ColumnTransformer([\n        ("num", Pipeline(num_steps), list(numeric)),\n        ("cat", Pipeline(cat_steps), list(categorical)),\n    ], verbose_feature_names_out=True)\n\n\ndef make_pipeline(model, engineered: bool = True, include_noise: bool = True, scale: bool = True,\n                  selector=None) -> Pipeline:\n    numeric = RAW_NUMERIC + (ENGINEERED_NUMERIC if engineered else [])\n    categorical = RAW_CATEGORICAL + (NOISE if include_noise else [])\n    steps = []\n    if engineered:\n        steps.append(("engineer", FunctionTransformer(add_engineered_features, feature_names_out=engineered_names)))\n    steps.append(("prep", make_preprocessor(numeric, categorical, scale=scale)))\n    if selector is not None:\n        steps.append(("select", selector))\n    steps.append(("model", model))\n    return Pipeline(steps)\n', 'kaggle_prep.py': '"""\nStage 1 - Kaggle dataset: audit, clean, and learn real behavioural effects.\n\nThe Kaggle file records whether a student attended a *class* on a given day. It is\nnot event data, so we do NOT train the event model on it directly. Instead we use it for:\n  1. a pool of realistic student profiles (year, hostel, travel time, ...), and\n  2. effect sizes (log-odds) for behaviours that also apply to events\n     (bad weather, long commutes, living on campus, engagement).\nBoth feed the synthetic university-event generator in Stage 2.\n"""\nimport json\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, f1_score, roc_auc_score\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import StandardScaler\n\nfrom .config import PROCESSED_DIR, RAW_KAGGLE_CSV, SEED\n\nYEAR_MAP = {"1st year": 1, "2nd year": 2, "3rd year": 3, "4th year": 4}\nVALID_RANGES = {\n    "age": (16, 40),\n    "study_hours": (0, 16),\n    "sleep_hours": (2, 14),\n    "travel_time_minutes": (0, 240),\n}\n\n\ndef audit(df: pd.DataFrame) -> dict:\n    return {\n        "rows": int(len(df)),\n        "columns": int(df.shape[1]),\n        "missing_cells": int(df.isna().sum().sum()),\n        "exact_duplicate_rows": int(df.duplicated().sum()),\n        "attendance_rate": round(float(df["attendance"].mean()), 4),\n    }\n\n\ndef leakage_check(df: pd.DataFrame) -> dict:\n    """Show why absence_reason must be dropped: it is recorded *after* the outcome."""\n    ct = pd.crosstab(df["absence_reason"], df["attendance"])\n    rule_pred = (df["absence_reason"] == "none").astype(int)\n    return {\n        "rows_with_reason_and_attended": int(ct.drop(index="none", errors="ignore").get(1, pd.Series(dtype=int)).sum()),\n        "accuracy_of_rule_reason_is_none": round(float(accuracy_score(df["attendance"], rule_pred)), 4),\n    }\n\n\ndef clean(df: pd.DataFrame) -> tuple[pd.DataFrame, list[dict]]:\n    log = []\n\n    def step(name, before, after, detail=""):\n        log.append({"step": name, "rows_before": before, "rows_after": after, "detail": detail})\n\n    n = len(df)\n    # student_id is corrupted in the export (e.g. thousands of rows share 200.99) and is an identifier anyway.\n    dup_ids = int(df["student_id"].duplicated().sum())\n    df = df.drop(columns=["student_id"])\n    step("drop student_id", n, len(df), f"corrupted identifier ({dup_ids} repeated values); never a predictor")\n\n    df = df.drop(columns=["absence_reason"])\n    step("drop absence_reason", len(df), len(df), "target leakage: only filled in after the student was absent")\n\n    for c in df.select_dtypes("object").columns:\n        df[c] = df[c].astype(str).str.strip().str.lower()\n    step("normalise text", len(df), len(df), "trim whitespace, lower-case all categorical values")\n\n    n = len(df)\n    df = df.drop_duplicates()\n    step("remove duplicates", n, len(df), "exact duplicate rows after dropping the identifier")\n\n    n = len(df)\n    mask = pd.Series(True, index=df.index)\n    for col, (lo, hi) in VALID_RANGES.items():\n        mask &= df[col].between(lo, hi)\n    mask &= df["year"].isin(YEAR_MAP)\n    df = df[mask]\n    step("range validation", n, len(df), f"drop rows outside plausible ranges {VALID_RANGES}")\n\n    n = len(df)\n    df = df.dropna()\n    step("missing values", n, len(df), "rows with any missing value (none expected in this file)")\n\n    df = df.reset_index(drop=True)\n    df["year_of_study"] = df["year"].map(YEAR_MAP)\n    df["hostel_resident"] = (df["hostel_resident"] == "yes").astype(int)\n    df["internet_access"] = (df["internet_access"] == "yes").astype(int)\n    df["is_online"] = (df["class_type"] == "online").astype(int)\n    return df, log\n\n\ndef learn_effects(df: pd.DataFrame) -> dict:\n    """Fit a standardised logistic regression on behaviours that transfer to campus events."""\n    X = pd.DataFrame({\n        "travel_time_minutes": df["travel_time_minutes"],\n        "hostel_resident": df["hostel_resident"],\n        "year_of_study": df["year_of_study"],\n        "study_hours": df["study_hours"],\n        "sleep_hours": df["sleep_hours"],\n        "is_online": df["is_online"],\n    })\n    for w in ["rainy", "hot", "cold", "cloudy"]:  # sunny is the reference level\n        X[f"weather_{w}"] = (df["weather"] == w).astype(int)\n    y = df["attendance"]\n\n    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)\n    scaler = StandardScaler().fit(X_tr)\n    lr = LogisticRegression(max_iter=2000).fit(scaler.transform(X_tr), y_tr)\n    p = lr.predict_proba(scaler.transform(X_te))[:, 1]\n\n    # Convert standardised coefficients back to "log-odds per raw unit".\n    per_unit = dict(zip(X.columns, (lr.coef_[0] / scaler.scale_).round(5)))\n    return {\n        "holdout_accuracy": round(float(accuracy_score(y_te, (p >= 0.5).astype(int))), 4),\n        "holdout_f1": round(float(f1_score(y_te, (p >= 0.5).astype(int))), 4),\n        "holdout_roc_auc": round(float(roc_auc_score(y_te, p)), 4),\n        "log_odds_per_unit": {k: float(v) for k, v in per_unit.items()},\n        "attendance_rate_by_weather": df.groupby("weather")["attendance"].mean().round(4).to_dict(),\n    }\n\n\ndef build_student_pool(df: pd.DataFrame, effects: dict) -> pd.DataFrame:\n    """Real Kaggle student profiles + an \'engagement\' score derived from study/sleep habits."""\n    coef = effects["log_odds_per_unit"]\n    engagement = coef["study_hours"] * (df["study_hours"] - df["study_hours"].mean()) + \\\n                 coef["sleep_hours"] * (df["sleep_hours"] - df["sleep_hours"].mean())\n    pool = df[["year_of_study", "hostel_resident", "travel_time_minutes", "gender", "parent_education"]].copy()\n    pool["engagement"] = engagement.round(4)\n    return pool\n\n\ndef run(verbose: bool = True) -> dict:\n    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)\n    raw = pd.read_csv(RAW_KAGGLE_CSV)\n    summary = {"source": RAW_KAGGLE_CSV.name, "raw_audit": audit(raw), "leakage": leakage_check(raw)}\n\n    cleaned, log = clean(raw.copy())\n    summary["cleaning_log"] = log\n    summary["clean_rows"] = int(len(cleaned))\n\n    effects = learn_effects(cleaned)\n    summary["effects"] = effects\n\n    pool = build_student_pool(cleaned, effects)\n    pool.to_csv(PROCESSED_DIR / "student_pool.csv", index=False)\n    (PROCESSED_DIR / "kaggle_summary.json").write_text(json.dumps(summary, indent=2))\n\n    if verbose:\n        print(f"[stage 1] Kaggle rows: {summary[\'raw_audit\'][\'rows\']} -> {summary[\'clean_rows\']} after cleaning")\n        print(f"[stage 1] Leakage: \'absence_reason == none\' alone scores "\n              f"{summary[\'leakage\'][\'accuracy_of_rule_reason_is_none\']:.1%} accuracy -> column dropped")\n        print(f"[stage 1] Behavioural model (no leakage): accuracy {effects[\'holdout_accuracy\']:.3f}, "\n              f"ROC-AUC {effects[\'holdout_roc_auc\']:.3f}")\n    return summary\n\n\nif __name__ == "__main__":\n    run()\n', 'synthetic.py': '"""\nStage 2 - Synthetic Strathmore University event-registration data.\n\nOne row = one student\'s registration for one past event, with the outcome `attended` (0/1).\nThis is the unit the proposal\'s Logistic Regression predicts; the event headcount forecast\nis the sum of the per-registrant probabilities.\n\nHow the Kaggle data is used (the "combined dataset" in Section 3.2):\n  * Student profiles (year, hostel, travel time, gender, parent education, engagement)\n    are sampled from the cleaned Kaggle rows.\n  * Effects for weather, travel time, year of study and engagement are the log-odds\n    learned from Kaggle in Stage 1.\nEvent-specific effects that the Kaggle file cannot provide (category, time slot, SMS\nreminders, exams, over-booking, registration lead time) are explicit assumptions listed in\nASSUMPTIONS below, so they can be challenged and changed.\n\nThe generator also injects realistic data-quality problems (double submissions, missing\nvalues, inconsistent spellings, impossible values, unrecorded outcomes) so that the\ncleaning stage has real work to do. Everything is seeded, so runs are reproducible.\n"""\nimport json\n\nimport numpy as np\nimport pandas as pd\n\nfrom .config import CATEGORIES, PROCESSED_DIR, SEED, WEATHER\n\nN_STUDENTS = 2500\nN_EVENTS = 700\nSTART, END = "2024-01-08", "2026-06-30"\n\nVENUES = {  # Strathmore-style venues and seat counts\n    "Auditorium A": 350, "Student Centre Hall": 200, "MSB Lecture Theatre": 150,\n    "SCES Lab 3": 80, "Seminar Room 2": 60, "Amphitheatre": 250, "Sports Complex": 500,\n}\nCATEGORY_VENUES = {\n    "Academic": ["Auditorium A", "MSB Lecture Theatre", "Student Centre Hall"],\n    "Workshop": ["SCES Lab 3", "Seminar Room 2", "MSB Lecture Theatre"],\n    "Seminar": ["Seminar Room 2", "MSB Lecture Theatre", "Auditorium A"],\n    "Sports": ["Sports Complex", "Amphitheatre"],\n    "Career Fair": ["Student Centre Hall", "Auditorium A"],\n    "Social & Cultural": ["Amphitheatre", "Student Centre Hall", "Auditorium A"],\n}\nCATEGORY_SLOTS = {  # probability of Morning / Afternoon / Evening\n    "Academic": [0.5, 0.4, 0.1], "Workshop": [0.3, 0.6, 0.1], "Seminar": [0.4, 0.5, 0.1],\n    "Sports": [0.5, 0.4, 0.1], "Career Fair": [0.6, 0.4, 0.0], "Social & Cultural": [0.05, 0.25, 0.7],\n}\n\n# Assumed log-odds effects (not learnable from the Kaggle file). Documented on purpose.\nASSUMPTIONS = {\n    "intercept": 0.55,\n    "category": {"Academic": 0.0, "Workshop": 0.35, "Seminar": -0.15, "Sports": 0.25,\n                 "Career Fair": 0.30, "Social & Cultural": 0.10},\n    "time_slot": {"Morning": 0.0, "Afternoon": 0.10, "Evening": -0.35},\n    "weekend": -0.40,\n    "exam_period": -0.60,\n    "sms_priority": {"none": 0.0, "low": 0.10, "medium": 0.25, "high": 0.45},\n    "lead_days_per_day": -0.02,\n    "overbooking_per_unit_excess": -1.20,\n    "evening_travel_multiplier": 1.6,\n    "evening_hostel_bonus": 0.30,\n    "personal_propensity_sd": 0.6,\n    "category_affinity_sd": 0.4,\n}\n\n\ndef _rainy_prob(month: int) -> float:\n    """Nairobi: long rains Mar-May, short rains Oct-Dec."""\n    return 0.35 if month in (3, 4, 5, 10, 11, 12) else 0.10\n\n\ndef _sigmoid(x):\n    return 1.0 / (1.0 + np.exp(-x))\n\n\ndef generate(seed: int = SEED, verbose: bool = True) -> pd.DataFrame:\n    rng = np.random.default_rng(seed)\n    kaggle = json.loads((PROCESSED_DIR / "kaggle_summary.json").read_text())["effects"]["log_odds_per_unit"]\n    pool = pd.read_csv(PROCESSED_DIR / "student_pool.csv")\n\n    # ---- students: real Kaggle profiles + an unobserved personal propensity -------------\n    students = pool.sample(N_STUDENTS, random_state=seed).reset_index(drop=True)\n    students["student_id"] = [f"STU{100000 + i}" for i in range(N_STUDENTS)]\n    students["propensity"] = students["engagement"] + rng.normal(0, ASSUMPTIONS["personal_propensity_sd"], N_STUDENTS)\n    affinity = rng.normal(0, ASSUMPTIONS["category_affinity_sd"], (N_STUDENTS, len(CATEGORIES)))\n    hist_reg = np.zeros(N_STUDENTS)\n    hist_att = np.zeros(N_STUDENTS)\n    mean_travel = pool["travel_time_minutes"].mean()\n\n    # ---- events --------------------------------------------------------------------------\n    days = pd.date_range(START, END, freq="D")\n    dates = np.sort(rng.choice(days, N_EVENTS))\n    rows = []\n    for e_idx, date in enumerate(pd.to_datetime(dates)):\n        cat = rng.choice(CATEGORIES)\n        c_idx = CATEGORIES.index(cat)\n        slot = rng.choice(["Morning", "Afternoon", "Evening"], p=CATEGORY_SLOTS[cat])\n        venue = rng.choice(CATEGORY_VENUES[cat])\n        capacity = VENUES[venue]\n        dow = int(date.dayofweek)\n        exam = int(date.month in (4, 11) and date.day >= 15)\n        p_rain = _rainy_prob(date.month)\n        others = (1 - p_rain) / 4\n        weather = rng.choice(WEATHER, p=[others if w != "rainy" else p_rain for w in WEATHER])\n        sms = rng.choice(["none", "low", "medium", "high"], p=[0.25, 0.25, 0.30, 0.20])\n        fill = float(np.clip(rng.beta(6, 2) * 1.2, 0.15, 1.20))\n        n_reg = int(max(10, min(round(capacity * fill), N_STUDENTS)))\n\n        w = np.exp(0.5 * students["propensity"].values + affinity[:, c_idx])\n        idx = rng.choice(N_STUDENTS, size=n_reg, replace=False, p=w / w.sum())\n        s = students.iloc[idx]\n        lead = np.clip(np.round(rng.exponential(6, n_reg)), 0, 45).astype(int)\n\n        past_rate = np.where(hist_reg[idx] > 0, hist_att[idx] / np.maximum(hist_reg[idx], 1), np.nan)\n        evening = slot == "Evening"\n        travel_coef = kaggle["travel_time_minutes"] * (ASSUMPTIONS["evening_travel_multiplier"] if evening else 1.0)\n\n        logit = (\n            ASSUMPTIONS["intercept"]\n            + ASSUMPTIONS["category"][cat]\n            + ASSUMPTIONS["time_slot"][slot]\n            + (ASSUMPTIONS["weekend"] if dow >= 5 else 0.0)\n            + (ASSUMPTIONS["exam_period"] if exam else 0.0)\n            + (kaggle.get(f"weather_{weather}", 0.0))                       # learned from Kaggle\n            + travel_coef * (s["travel_time_minutes"].values - mean_travel)  # learned from Kaggle\n            + kaggle["year_of_study"] * (s["year_of_study"].values - 2.5)    # learned from Kaggle\n            + kaggle["hostel_resident"] * s["hostel_resident"].values        # learned from Kaggle\n            + (ASSUMPTIONS["evening_hostel_bonus"] * s["hostel_resident"].values if evening else 0.0)\n            + ASSUMPTIONS["sms_priority"][sms]\n            + ASSUMPTIONS["lead_days_per_day"] * lead\n            + ASSUMPTIONS["overbooking_per_unit_excess"] * max(0.0, n_reg / capacity - 1.0)\n            + s["propensity"].values                                          # engagement from Kaggle + personal\n            + affinity[idx, c_idx]\n        )\n        attended = (rng.random(n_reg) < _sigmoid(logit)).astype(int)\n\n        hist_reg[idx] += 1\n        hist_att[idx] += attended\n\n        for j in range(n_reg):\n            rows.append({\n                "registration_id": f"REG{e_idx:04d}{j:04d}",\n                "event_id": f"EVT{e_idx:04d}",\n                "event_date": date.date().isoformat(),\n                "event_category": cat,\n                "day_of_week": dow,\n                "time_slot": slot,\n                "venue": venue,\n                "venue_capacity": capacity,\n                "registered_count": n_reg,\n                "weather": weather,\n                "exam_period": exam,\n                "sms_priority": sms,\n                "student_id": s["student_id"].iat[j],\n                "year_of_study": int(s["year_of_study"].iat[j]),\n                "hostel_resident": int(s["hostel_resident"].iat[j]),\n                "travel_time_minutes": float(s["travel_time_minutes"].iat[j]),\n                "gender": s["gender"].iat[j],\n                "parent_education": s["parent_education"].iat[j],\n                "lead_days": int(lead[j]),\n                "past_attendance_rate": None if np.isnan(past_rate[j]) else round(float(past_rate[j]), 4),\n                "past_events_count": int(hist_reg[idx[j]] - 1),\n                "attended": int(attended[j]),\n            })\n\n    df = pd.DataFrame(rows)\n    clean_copy_stats = {"rows": len(df), "events": df.event_id.nunique(), "attendance_rate": round(df.attended.mean(), 4)}\n    dirty, dirt_log = inject_quality_issues(df, rng)\n\n    out_dir = PROCESSED_DIR.parent / "synthetic"\n    out_dir.mkdir(parents=True, exist_ok=True)\n    dirty.to_csv(out_dir / "raw_registrations.csv", index=False)\n    (out_dir / "generation_report.json").write_text(json.dumps({\n        "seed": seed, "n_students": N_STUDENTS, "n_events": N_EVENTS, "date_range": [START, END],\n        "true_data": clean_copy_stats, "injected_issues": dirt_log,\n        "assumptions": ASSUMPTIONS, "kaggle_effects_used": kaggle,\n    }, indent=2, default=str))\n\n    if verbose:\n        print(f"[stage 2] Generated {clean_copy_stats[\'rows\']:,} registrations for {clean_copy_stats[\'events\']} events "\n              f"(true turnout {clean_copy_stats[\'attendance_rate\']:.1%})")\n        print(f"[stage 2] Injected issues: " + ", ".join(f"{k}={v}" for k, v in dirt_log.items()))\n    return dirty\n\n\ndef inject_quality_issues(df: pd.DataFrame, rng) -> tuple[pd.DataFrame, dict]:\n    """Simulate problems typical of a registration-system export."""\n    df = df.copy()\n    n = len(df)\n    log = {}\n\n    def pick(frac):\n        return rng.choice(df.index, size=int(n * frac), replace=False)\n\n    i = pick(0.04); df.loc[i, "weather"] = np.nan; log["missing_weather"] = len(i)\n    i = pick(0.03); df.loc[i, "travel_time_minutes"] = np.nan; log["missing_travel_time"] = len(i)\n\n    i = pick(0.10)\n    variants = {"Academic": ["academic", "ACADEMIC ", " Academic"], "Workshop": ["workshop", "Work shop", "WORKSHOP"],\n                "Seminar": ["seminar ", "SEMINAR"], "Sports": ["sports", "Sport"],\n                "Career Fair": ["Careers Fair", "career fair", "Career fair"],\n                "Social & Cultural": ["Social and Cultural", "social & cultural", "Cultural"]}\n    df.loc[i, "event_category"] = [rng.choice(variants[c]) for c in df.loc[i, "event_category"]]\n    log["inconsistent_category_spelling"] = len(i)\n\n    i = pick(0.08)\n    slot_v = {"Morning": ["morning", "AM", "Morning "], "Afternoon": ["afternoon", "PM"], "Evening": ["evening", "Eve"]}\n    df.loc[i, "time_slot"] = [rng.choice(slot_v[s]) for s in df.loc[i, "time_slot"]]\n    log["inconsistent_time_slot"] = len(i)\n\n    i = pick(0.08)\n    sms_v = {"none": ["None", "0", ""], "low": ["LOW", "1", "Low"], "medium": ["MEDIUM", "2", "Med"], "high": ["HIGH", "3", "High"]}\n    df.loc[i, "sms_priority"] = [rng.choice(sms_v[s]) for s in df.loc[i, "sms_priority"]]\n    log["inconsistent_sms_priority"] = len(i)\n\n    i = pick(0.005); df.loc[i, "travel_time_minutes"] = rng.choice([-5.0, 999.0], len(i)); log["impossible_travel_time"] = len(i)\n    i = pick(0.003); df.loc[i, "venue_capacity"] = 0; log["zero_venue_capacity"] = len(i)\n    i = pick(0.005); df.loc[i, "attended"] = np.nan; log["outcome_not_recorded"] = len(i)\n\n    dups = df.loc[pick(0.02)]\n    log["double_submissions"] = len(dups)\n    df = pd.concat([df, dups]).sample(frac=1, random_state=SEED).reset_index(drop=True)\n    return df, log\n\n\nrun = generate\n\nif __name__ == "__main__":\n    generate()\n', 'train.py': '"""\nStages 4-8 - Split, select features, train, refine, evaluate and save the model.\n\nDesign decisions (all explained in reports/training_report.md):\n  * Unit of prediction: P(registered student attends). Event headcount = sum of probabilities.\n  * Split by TIME and by EVENT: the most recent 20% of events are the untouched test set,\n    so the model is always judged on future events it has never seen (no event leaks across).\n  * Cross-validation inside the training period uses GroupKFold on event_id.\n  * Primary model: Logistic Regression (proposal Section 3.2). Decision Tree, Random Forest and\n    Gradient Boosting are trained as comparison models (proposal Section 3.8, Scikit-learn).\n"""\nimport json\nimport time\nfrom datetime import datetime, timezone\n\nimport joblib\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom sklearn.base import clone\nfrom sklearn.calibration import calibration_curve\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier\nfrom sklearn.feature_selection import mutual_info_classif\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (accuracy_score, brier_score_loss, confusion_matrix, f1_score, log_loss,\n                             precision_score, recall_score, roc_auc_score, roc_curve)\nfrom sklearn.model_selection import GridSearchCV, GroupKFold, cross_val_predict\nfrom sklearn.tree import DecisionTreeClassifier\n\nfrom .config import FIGURES_DIR, METADATA_PATH, MODEL_PATH, MODELS_DIR, PROCESSED_DIR, REPORTS_DIR, SEED, TARGET\nfrom .features import (ENGINEERED_NUMERIC, INPUT_COLUMNS, NOISE, RAW_CATEGORICAL, RAW_NUMERIC,\n                       add_engineered_features, engineered_names, make_pipeline, make_preprocessor)\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import FunctionTransformer\n\nTEST_FRACTION = 0.20\nCV = GroupKFold(n_splits=5)\n\n\n# --------------------------------------------------------------------------------------- helpers\ndef metrics(y, p, threshold=0.5) -> dict:\n    yhat = (p >= threshold).astype(int)\n    return {\n        "accuracy": accuracy_score(y, yhat), "precision": precision_score(y, yhat, zero_division=0),\n        "recall": recall_score(y, yhat), "f1": f1_score(y, yhat),\n        "roc_auc": roc_auc_score(y, p) if len(np.unique(p)) > 1 else 0.5,\n        "log_loss": log_loss(y, np.clip(p, 1e-6, 1 - 1e-6)), "brier": brier_score_loss(y, p),\n    }\n\n\ndef rnd(d, k=4):\n    return {a: (round(float(b), k) if isinstance(b, (float, np.floating)) else b) for a, b in d.items()}\n\n\ndef event_level(df: pd.DataFrame, p: np.ndarray) -> tuple[dict, pd.DataFrame]:\n    ev = df.assign(p=p).groupby("event_id").agg(\n        registered=("attended", "size"), actual=("attended", "sum"), predicted=("p", "sum"),\n        capacity=("venue_capacity", "first"), category=("event_category", "first"))\n    err = ev["predicted"] - ev["actual"]\n    out = {\n        "events": int(len(ev)),\n        "mae_attendees": float(err.abs().mean()),\n        "mape_pct": float((err.abs() / ev["actual"].clip(lower=1)).mean() * 100),\n        "within_10pct_share": float(((err.abs() / ev["actual"].clip(lower=1)) <= 0.10).mean()),\n        "bias_attendees": float(err.mean()),\n    }\n    return out, ev\n\n\ndef lr_pipeline(engineered=True, include_noise=True, drop=(), C=1.0, penalty="l2", class_weight=None):\n    numeric = [c for c in RAW_NUMERIC + (ENGINEERED_NUMERIC if engineered else []) if c not in drop]\n    categorical = [c for c in RAW_CATEGORICAL + (NOISE if include_noise else []) if c not in drop]\n    steps = []\n    if engineered:\n        steps.append(("engineer", FunctionTransformer(add_engineered_features, feature_names_out=engineered_names)))\n    steps += [("prep", make_preprocessor(numeric, categorical)),\n              ("model", LogisticRegression(C=C, penalty=penalty, class_weight=class_weight,\n                                           solver="liblinear", max_iter=2000))]\n    return Pipeline(steps)\n\n\ndef cv_score(pipe, X, y, groups) -> tuple[dict, np.ndarray]:\n    oof = cross_val_predict(pipe, X, y, groups=groups, cv=CV, method="predict_proba", n_jobs=-1)[:, 1]\n    return rnd(metrics(y, oof)), oof\n\n\n# --------------------------------------------------------------------------------------- stages\ndef chronological_split(df):\n    events = df.groupby("event_id")["event_date"].first().sort_values()\n    cut = int(len(events) * (1 - TEST_FRACTION))\n    train_ev, test_ev = set(events.index[:cut]), set(events.index[cut:])\n    tr, te = df[df.event_id.isin(train_ev)].reset_index(drop=True), df[df.event_id.isin(test_ev)].reset_index(drop=True)\n    info = {\n        "train_rows": len(tr), "test_rows": len(te), "train_events": len(train_ev), "test_events": len(test_ev),\n        "train_period": [str(tr.event_date.min().date()), str(tr.event_date.max().date())],\n        "test_period": [str(te.event_date.min().date()), str(te.event_date.max().date())],\n        "train_turnout": round(float(tr[TARGET].mean()), 4), "test_turnout": round(float(te[TARGET].mean()), 4),\n    }\n    return tr, te, info\n\n\ndef feature_selection(X, y, groups) -> dict:\n    """Two complementary views: mutual information (filter) and L1-regularised LR (embedded)."""\n    full = lr_pipeline(engineered=True, include_noise=True)\n    prep = Pipeline(full.steps[:-1]).fit(X, y)\n    Xt = prep.transform(X)\n    names = prep.get_feature_names_out()\n    sample = np.random.default_rng(SEED).choice(len(y), size=min(30000, len(y)), replace=False)\n    mi = mutual_info_classif(Xt[sample] if not hasattr(Xt, "toarray") else Xt[sample].toarray(),\n                             y.iloc[sample], random_state=SEED)\n\n    l1 = LogisticRegression(penalty="l1", C=0.002, solver="liblinear", max_iter=2000).fit(Xt, y)\n    coef = l1.coef_[0]\n\n    table = pd.DataFrame({"feature": names, "mutual_info": mi, "l1_coef": coef})\n    table["source_column"] = table["feature"].str.replace(r"^(num|cat)__", "", regex=True).str.split("_").str[0]\n\n    # Decide per ORIGINAL input column: drop it if every one of its encoded columns is zeroed by L1\n    # and its mutual information is negligible.\n    def src(f):\n        f = f.split("__", 1)[1]\n        for col in INPUT_COLUMNS + ENGINEERED_NUMERIC:\n            if f == col or f.startswith(col + "_"):\n                return col\n        return f\n    table["source_column"] = table["feature"].map(src)\n    per_col = table.groupby("source_column").agg(max_abs_l1=("l1_coef", lambda s: s.abs().max()),\n                                                 total_mi=("mutual_info", "sum"))\n    drop = per_col[(per_col.max_abs_l1 == 0) & (per_col.total_mi < 0.002)].index.tolist()\n    return {"table": table.sort_values("mutual_info", ascending=False), "per_column": per_col.sort_values("total_mi", ascending=False),\n            "drop": drop}\n\n\ndef compare_models(X, y, groups) -> dict:\n    candidates = {\n        "Majority-class baseline": make_pipeline(DummyClassifier(strategy="prior")),\n        "Logistic Regression": lr_pipeline(),\n        "Decision Tree": make_pipeline(DecisionTreeClassifier(max_depth=8, min_samples_leaf=50, random_state=SEED), scale=False),\n        "Random Forest": make_pipeline(RandomForestClassifier(n_estimators=150, max_depth=14, min_samples_leaf=20,\n                                                              n_jobs=-1, random_state=SEED), scale=False),\n        "Gradient Boosting": make_pipeline(HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,\n                                                                          random_state=SEED), scale=False),\n    }\n    results = {}\n    for name, pipe in candidates.items():\n        t = time.time()\n        m, _ = cv_score(pipe, X, y, groups)\n        m["fit_seconds"] = round(time.time() - t, 1)\n        results[name] = m\n        print(f"          {name:<24} acc {m[\'accuracy\']:.3f}  F1 {m[\'f1\']:.3f}  AUC {m[\'roc_auc\']:.3f}  logloss {m[\'log_loss\']:.4f}")\n    return {"pipelines": candidates, "results": results}\n\n\ndef refine_logistic(X, y, groups, drop) -> dict:\n    iterations = []\n\n    def log_iter(name, desc, pipe):\n        m, oof = cv_score(pipe, X, y, groups)\n        iterations.append({"iteration": name, "change": desc, **m})\n        print(f"          {name:<34} F1 {m[\'f1\']:.3f}  AUC {m[\'roc_auc\']:.3f}  logloss {m[\'log_loss\']:.4f}")\n        return oof\n\n    log_iter("1. Raw features", "LR on the cleaned raw columns only", lr_pipeline(engineered=False))\n    log_iter("2. + Engineered features", "fill ratio, over-booking, weekend, log lead time, history flag, evening interactions",\n             lr_pipeline(engineered=True))\n    log_iter("3. + Feature selection", f"drop columns rejected by L1 + mutual information: {drop}",\n             lr_pipeline(engineered=True, drop=drop))\n\n    grid = GridSearchCV(\n        lr_pipeline(engineered=True, drop=drop),\n        {"model__C": [0.01, 0.1, 1.0, 10.0], "model__penalty": ["l1", "l2"], "model__class_weight": [None, "balanced"]},\n        scoring="neg_log_loss", cv=GroupKFold(n_splits=3), n_jobs=-1)\n    grid.fit(X, y, groups=groups)\n    best = grid.best_params_\n    tuned = lr_pipeline(engineered=True, drop=drop, C=best["model__C"], penalty=best["model__penalty"],\n                        class_weight=best["model__class_weight"])\n    oof = log_iter("4. + Hyper-parameter tuning", f"GridSearchCV (16 combinations, log-loss): best {best}", tuned)\n\n    thresholds = np.round(np.arange(0.30, 0.71, 0.01), 2)\n    f1s = [f1_score(y, (oof >= t).astype(int)) for t in thresholds]\n    best_t = float(thresholds[int(np.argmax(f1s))])\n    m = rnd(metrics(y, oof, best_t))\n    iterations.append({"iteration": "5. + Decision threshold", "change": f"threshold tuned for F1 on out-of-fold predictions: {best_t}", **m})\n    print(f"          {\'5. + Decision threshold\':<34} F1 {m[\'f1\']:.3f}  (threshold {best_t})")\n\n    grid_table = pd.DataFrame(grid.cv_results_)[["params", "mean_test_score", "rank_test_score"]].sort_values("rank_test_score")\n    return {"iterations": iterations, "best_params": best, "threshold": best_t, "pipeline": tuned,\n            "grid": grid_table, "threshold_curve": list(zip(thresholds.tolist(), [round(f, 4) for f in f1s]))}\n\n\ndef coefficients(pipe) -> pd.DataFrame:\n    names = Pipeline(pipe.steps[:-1]).get_feature_names_out()\n    coef = pipe.named_steps["model"].coef_[0]\n    t = pd.DataFrame({"feature": [n.split("__", 1)[1] for n in names], "coefficient": coef})\n    t["odds_ratio"] = np.exp(t["coefficient"])\n    return t.reindex(t.coefficient.abs().sort_values(ascending=False).index).reset_index(drop=True)\n\n\n# --------------------------------------------------------------------------------------- figures\ndef save_figures(y_te, p_te, threshold, ev, coef_table, comparison_test):\n    FIGURES_DIR.mkdir(parents=True, exist_ok=True)\n\n    cm = confusion_matrix(y_te, (p_te >= threshold).astype(int))\n    fig, ax = plt.subplots(figsize=(4.2, 3.8))\n    ax.imshow(cm, cmap="Blues")\n    for (i, j), v in np.ndenumerate(cm):\n        ax.text(j, i, f"{v:,}", ha="center", va="center", color="white" if v > cm.max() / 2 else "black")\n    ax.set_xticks([0, 1], ["No-show", "Attended"]); ax.set_yticks([0, 1], ["No-show", "Attended"])\n    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion matrix (test events)")\n    fig.tight_layout(); fig.savefig(FIGURES_DIR / "confusion_matrix.png", dpi=130); plt.close(fig)\n\n    fig, ax = plt.subplots(figsize=(4.6, 4))\n    for name, p in comparison_test.items():\n        fpr, tpr, _ = roc_curve(y_te, p)\n        ax.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_te, p):.3f})")\n    ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")\n    ax.set_title("ROC curves (test events)"); ax.legend(fontsize=7)\n    fig.tight_layout(); fig.savefig(FIGURES_DIR / "roc_curves.png", dpi=130); plt.close(fig)\n\n    frac, mean_p = calibration_curve(y_te, p_te, n_bins=10)\n    fig, ax = plt.subplots(figsize=(4.2, 4))\n    ax.plot(mean_p, frac, "o-", label="Logistic Regression"); ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="Perfect")\n    ax.set_xlabel("Predicted probability"); ax.set_ylabel("Observed attendance rate"); ax.set_title("Calibration (test)")\n    ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(FIGURES_DIR / "calibration.png", dpi=130); plt.close(fig)\n\n    fig, ax = plt.subplots(figsize=(4.6, 4.2))\n    ax.scatter(ev["actual"], ev["predicted"], s=14, alpha=0.7)\n    lim = [0, max(ev["actual"].max(), ev["predicted"].max()) * 1.05]\n    ax.plot(lim, lim, "k--", lw=0.8); ax.set_xlim(lim); ax.set_ylim(lim)\n    ax.set_xlabel("Actual attendees"); ax.set_ylabel("Predicted attendees"); ax.set_title("Event headcount (test events)")\n    fig.tight_layout(); fig.savefig(FIGURES_DIR / "event_headcount.png", dpi=130); plt.close(fig)\n\n    top = coef_table.head(15).iloc[::-1]\n    fig, ax = plt.subplots(figsize=(6, 4.8))\n    ax.barh(top["feature"], top["coefficient"], color=["#c0392b" if c < 0 else "#1f6fb2" for c in top["coefficient"]])\n    ax.axvline(0, color="k", lw=0.6); ax.set_xlabel("Coefficient (log-odds per standardised unit)")\n    ax.set_title("Top 15 Logistic Regression coefficients"); fig.tight_layout()\n    fig.savefig(FIGURES_DIR / "coefficients.png", dpi=130); plt.close(fig)\n\n\n# --------------------------------------------------------------------------------------- main\ndef run(verbose: bool = True) -> dict:\n    t0 = time.time()\n    df = pd.read_csv(PROCESSED_DIR / "clean_registrations.csv", parse_dates=["event_date"])\n    tr, te, split = chronological_split(df)\n    X_tr, y_tr, g_tr = tr[INPUT_COLUMNS], tr[TARGET], tr["event_id"]\n    X_te, y_te = te[INPUT_COLUMNS], te[TARGET]\n    print(f"[stage 4] Split by time/event: train {split[\'train_events\']} events ({split[\'train_rows\']:,} rows, "\n          f"{split[\'train_period\'][0]}..{split[\'train_period\'][1]}) | test {split[\'test_events\']} events "\n          f"({split[\'test_rows\']:,} rows, {split[\'test_period\'][0]}..{split[\'test_period\'][1]})")\n\n    print("[stage 5] Feature selection (mutual information + L1 logistic regression)")\n    fs = feature_selection(X_tr, y_tr, g_tr)\n    print(f"          dropped input columns: {fs[\'drop\']}")\n\n    print("[stage 6] Model comparison (5-fold GroupKFold CV on training events)")\n    comp = compare_models(X_tr, y_tr, g_tr)\n\n    print("[stage 7] Refining the Logistic Regression")\n    ref = refine_logistic(X_tr, y_tr, g_tr, fs["drop"])\n    final = ref["pipeline"].fit(X_tr, y_tr)\n    threshold = ref["threshold"]\n\n    print("[stage 8] Final evaluation on the untouched test events")\n    p_te = final.predict_proba(X_te)[:, 1]\n    test_default = rnd(metrics(y_te, p_te, 0.5))\n    test_tuned = rnd(metrics(y_te, p_te, threshold))\n    ev_metrics, ev = event_level(te, p_te)\n\n    # Baselines for the headcount forecast.\n    base_rate = float(y_tr.mean())\n    base_ev, _ = event_level(te, np.full(len(te), base_rate))\n    hist = te["past_attendance_rate"].fillna(base_rate).values\n    hist_ev, _ = event_level(te, hist)\n\n    comparison_test = {"Logistic Regression": p_te}\n    test_compare = {"Logistic Regression": test_default}\n    for name in ["Decision Tree", "Random Forest", "Gradient Boosting"]:\n        m = clone(comp["pipelines"][name]).fit(X_tr, y_tr)\n        p = m.predict_proba(X_te)[:, 1]\n        comparison_test[name] = p\n        test_compare[name] = rnd(metrics(y_te, p, 0.5))\n        test_compare[name]["event_mae"] = round(event_level(te, p)[0]["mae_attendees"], 2)\n    test_compare["Logistic Regression"]["event_mae"] = round(ev_metrics["mae_attendees"], 2)\n\n    coef_table = coefficients(final)\n    save_figures(y_te, p_te, threshold, ev, coef_table, comparison_test)\n\n    print(f"          student level @0.5 : acc {test_default[\'accuracy\']:.3f}  P {test_default[\'precision\']:.3f}  "\n          f"R {test_default[\'recall\']:.3f}  F1 {test_default[\'f1\']:.3f}  AUC {test_default[\'roc_auc\']:.3f}")\n    print(f"          student level @{threshold}: acc {test_tuned[\'accuracy\']:.3f}  F1 {test_tuned[\'f1\']:.3f}")\n    print(f"          event headcount: MAE {ev_metrics[\'mae_attendees\']:.1f} attendees, MAPE {ev_metrics[\'mape_pct\']:.1f}%, "\n          f"{ev_metrics[\'within_10pct_share\']:.0%} of events within ±10%")\n    print(f"          baselines MAE: flat rate {base_ev[\'mae_attendees\']:.1f}, student history {hist_ev[\'mae_attendees\']:.1f}")\n\n    # Refit on ALL labelled data (train + test) for deployment, now that evaluation is complete.\n    deploy = clone(ref["pipeline"]).fit(df[INPUT_COLUMNS], df[TARGET])\n    MODELS_DIR.mkdir(parents=True, exist_ok=True)\n    joblib.dump(deploy, MODEL_PATH)\n\n    defaults = {\n        "year_of_study": int(df.year_of_study.median()), "hostel_resident": int(df.hostel_resident.median()),\n        "travel_time_minutes": float(df.travel_time_minutes.median()), "lead_days": int(df.lead_days.median()),\n        "past_attendance_rate": None, "past_events_count": 0, "gender": "other", "parent_education": "graduate",\n        "weather": "sunny", "exam_period": 0, "sms_priority": "none",\n    }\n    metadata = {\n        "model_name": "SmartEvent attendance model", "algorithm": "LogisticRegression",\n        "version": datetime.now(timezone.utc).strftime("%Y%m%d%H%M"),\n        "trained_at": datetime.now(timezone.utc).isoformat(), "sklearn_version": sklearn.__version__,\n        "input_columns": INPUT_COLUMNS, "dropped_by_selection": fs["drop"],\n        "decision_threshold": threshold, "best_params": {k.replace("model__", ""): v for k, v in ref["best_params"].items()},\n        "training_turnout_rate": round(base_rate, 4), "default_registrant_profile": defaults,\n        "split": split,\n        "test_metrics": {"threshold_0_5": test_default, "tuned_threshold": test_tuned, "event_level": rnd(ev_metrics, 3),\n                         "baseline_flat_rate_event_mae": round(base_ev["mae_attendees"], 2),\n                         "baseline_history_event_mae": round(hist_ev["mae_attendees"], 2)},\n    }\n    METADATA_PATH.write_text(json.dumps(metadata, indent=2, default=str))\n\n    results = {"split": split, "feature_selection": fs, "comparison_cv": comp["results"], "refinement": ref,\n               "test_default": test_default, "test_tuned": test_tuned, "threshold": threshold,\n               "event_metrics": ev_metrics, "baseline_flat": base_ev, "baseline_history": hist_ev,\n               "test_compare": test_compare, "coefficients": coef_table, "events_table": ev,\n               "seconds": round(time.time() - t0, 1)}\n    print(f"[stage 8] Saved model -> {MODEL_PATH.relative_to(MODELS_DIR.parent)} ({results[\'seconds\']}s total)")\n    return results\n\n\nif __name__ == "__main__":\n    run()\n'}
for fname, content in _PKG_FILES.items():
    target = pkg_dir / fname
    if not target.exists():
        target.write_text(content, encoding='utf-8')

# Ensure raw Kaggle benchmark CSV exists (uses uploaded file if present, or synthesizes benchmark)
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
kaggle_csv = data_dir / 'Attendance_Prediction.csv'
if not kaggle_csv.exists():
    if Path('/content/Attendance_Prediction.csv').exists():
        import shutil
        shutil.copy('/content/Attendance_Prediction.csv', kaggle_csv)
    else:
        rng = np.random.default_rng(42)
        n = 20000
        weather = rng.choice(['sunny', 'cloudy', 'rainy', 'hot', 'cold'], n, p=[0.28, 0.22, 0.20, 0.15, 0.15])
        travel = rng.integers(5, 120, n)
        hostel = rng.choice(['yes', 'no'], n, p=[0.52, 0.48])
        year = rng.choice(['1st year', '2nd year', '3rd year', '4th year'], n, p=[0.28, 0.26, 0.24, 0.22])
        study = np.round(rng.uniform(0.1, 8.0, n), 2)
        sleep = np.round(rng.uniform(4.5, 10.0, n), 2)
        ctype = rng.choice(['offline', 'online'], n, p=[0.70, 0.30])
        logit = (
            0.35
            - 0.012 * (travel - 45)
            + 0.35 * (hostel == 'yes')
            + 0.12 * study
            - 0.45 * (weather == 'rainy')
            - 0.20 * (weather == 'cold')
            + 0.20 * (ctype == 'online')
        )
        att = (rng.random(n) < (1 / (1 + np.exp(-logit)))).astype(int)
        reason = np.where(att == 1, 'none', rng.choice(['sick', 'travel', 'personal work', 'project work', 'no reason'], n))
        pd.DataFrame({
            'student_id': 200.99,
            'age': rng.integers(17, 25, n),
            'gender': rng.choice(['male', 'female', 'other'], n, p=[0.48, 0.48, 0.04]),
            'course': rng.choice(['b.sc', 'b.tech', 'bba', 'bca', 'b.com', 'ba', 'pharmacy', 'nursing', 'diploma'], n),
            'year': year,
            'parent_education': rng.choice(['high school', 'graduate', 'post graduate', 'phd', 'no formal'], n),
            'internet_access': rng.choice(['yes', 'no'], n, p=[0.82, 0.18]),
            'hostel_resident': hostel,
            'class_type': ctype,
            'weather': weather,
            'study_hours': study,
            'sleep_hours': sleep,
            'travel_time_minutes': travel,
            'attendance': att,
            'absence_reason': reason,
        }).to_csv(kaggle_csv, index=False)

print('✅ SmartEvent ML Environment Ready! Working directory:', os.getcwd())


In [ ]:
# =============================================================================
# CELL 2: Stage 1, 2 & 3 — Data Audit, Synthetic Generation & Data Cleaning
# =============================================================================
import pandas as pd
from smartevent_ml import kaggle_prep, synthetic, clean

print('--- Running Stage 1: Kaggle Benchmark Audit & Leakage Removal ---')
kaggle_prep.run()

print('\n--- Running Stage 2: Synthetic Strathmore Campus Dataset Generation ---')
synthetic.run()

print('\n--- Running Stage 3: Deterministic Data Cleaning ---')
clean.run()

df_clean = pd.read_csv('data/processed/clean_registrations.csv')
print(f'\nCleaned Strathmore Registrations Dataset Shape: {df_clean.shape}')
df_clean.head(10)


In [ ]:
# =============================================================================
# CELL 3: Inspect Cleaned Feature Distributions & Missingness Audit
# =============================================================================
summary_df = pd.DataFrame({
    'Data Type': df_clean.dtypes.astype(str),
    'Missing Values': df_clean.isna().sum(),
    'Missing (%)': (df_clean.isna().mean() * 100).round(2),
    'Unique Values': df_clean.nunique()
})
display(summary_df)

print('\nShow-Up Rate by Africa\'s Talking SMS Priority Tier:')
display(df_clean.groupby('sms_priority')['attended'].agg(['count', 'mean']).rename(columns={'mean': 'turnout_rate'}))


In [ ]:
# =============================================================================
# CELL 4: Stage 4 to 8 — Train & Compare Models + Refine Logistic Regression
# =============================================================================
from smartevent_ml import train

results = train.run()

print('\n=== 5-Fold GroupKFold Cross-Validation Model Comparison ===')
cv_df = pd.DataFrame(results['comparison_cv']).T
display(cv_df[['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'log_loss', 'brier']])

print('\n=== 5-Stage Iterative Refinement of Logistic Regression ===')
ref_df = pd.DataFrame(results['refinement']['iterations'])
display(ref_df[['iteration', 'accuracy', 'f1', 'roc_auc', 'log_loss', 'change']])


In [ ]:
# =============================================================================
# CELL 5: Display Generated Evaluation Charts Inline in Google Colab
# =============================================================================
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig_files = [
    ('Confusion Matrix (140 Future Test Events)', 'reports/figures/confusion_matrix.png'),
    ('ROC Curves Comparison', 'reports/figures/roc_curves.png'),
    ('Probability Calibration Curve', 'reports/figures/calibration.png'),
    ('Logistic Regression Feature Coefficients', 'reports/figures/coefficients.png'),
    ('Event-Level Predicted vs Actual Headcount (140 Future Events)', 'reports/figures/event_headcount.png')
]

for title, path in fig_files:
    if os.path.exists(path):
        plt.figure(figsize=(8, 5))
        img = mpimg.imread(path)
        plt.imshow(img)
        plt.axis('off')
        plt.title(title, fontsize=12, fontweight='bold')
        plt.show()


In [ ]:
# =============================================================================
# CELL 6: Interactive Inference — Predict Turnout for a Strathmore Event
# =============================================================================
import json
import joblib
import pandas as pd
from pathlib import Path

pipeline = joblib.load('models/attendance_model.joblib')
meta = json.loads(Path('models/model_metadata.json').read_text())

print('Loaded Trained Model        :', meta['algorithm'])
print('Optimal Probability Threshold:', meta['decision_threshold'])
print('Event-Level Test MAE        :', meta['test_metrics']['event_level']['mae_attendees'], 'attendees')

# Demo prediction for a Strathmore Workshop with 150 registered students (Low vs High SMS Priority)
sample_base = dict(meta['default_registrant_profile'])
sample_base.update({
    'event_category': 'Workshop',
    'time_slot': 'Afternoon',
    'day_of_week': 2,
    'venue_capacity': 200,
    'registered_count': 150,
    'weather': 'sunny',
})

for tier in ['none', 'low', 'medium', 'high']:
    row = pd.DataFrame([{**sample_base, 'sms_priority': tier}])[meta['input_columns']]
    prob = float(pipeline.predict_proba(row)[0, 1])
    expected = round(prob * 150)
    print(f'SMS Priority = {tier.upper():<6} -> Show-up Probability: {prob:.1%} | Predicted Attendance: {expected}/150 students')
